# YOLOv8 NPH Detection --- Data Preparation Pipeline

**Author:** Matheus Machado Rech  
**Version:** 1.0.0  
**License:** Research use only --- not for clinical diagnosis

## Description

This notebook prepares labeled training data for a YOLOv8 model to detect Normal Pressure Hydrocephalus (NPH) features on axial CT brain slices. The pipeline:

1. Downloads public brain CT datasets (or generates synthetic volumes for testing)
2. Extracts brain-windowed axial slices (WL=40, WW=80) as PNGs
3. Generates YOLO bounding box labels using HU-based ventricle segmentation
4. Organizes data into YOLO training format (train/val split by case)
5. Trains YOLOv8 with CT-safe augmentation constraints
6. Validates and exports the model (ONNX)

### YOLO Classes

| ID | Class | Auto-labeled | Notes |
|----|-------|:---:|-------|
| 0 | `ventricle` | Yes | From NIfTI ventricle masks or HU thresholding |
| 1 | `sylvian_fissure` | No | Requires manual annotation |
| 2 | `tight_convexity` | No | Requires manual annotation |
| 3 | `pvh` | No | Requires manual annotation |
| 4 | `skull_inner` | Yes | From bone thresholding (HU > 300) |

### CT-Safe Augmentation Constraints

- `fliplr=0.0` --- laterality is clinically meaningful (left/right brain distinction)
- `mosaic=0.0` --- destroys spatial anatomy context
- `mixup=0.0` --- destroys spatial anatomy context
- `hsv_h=0.0, hsv_s=0.0` --- CT is grayscale, hue/saturation are meaningless
- `flipud=0.5` --- axial symmetry is safe

---

## Table of Contents

1. [Install Dependencies](#1-install-dependencies)
2. [Download Public Brain CT Data](#2-download-public-brain-ct-data)
3. [Extract Axial Slices as Brain-Windowed PNGs](#3-extract-axial-slices-as-brain-windowed-pngs)
4. [Generate YOLO Bounding Box Labels](#4-generate-yolo-bounding-box-labels)
5. [Organize into YOLO Dataset Format](#5-organize-into-yolo-dataset-format)
6. [Train YOLOv8 (Optional)](#6-train-yolov8-optional)
7. [Validate and Export](#7-validate-and-export)
8. [Next Steps](#8-next-steps)

In [ ]:
# ============================================================
# Install Dependencies
# ============================================================
# Run this cell once per Colab session. Restart runtime if prompted.

!pip install -q ultralytics nibabel SimpleITK datasets scikit-image scipy opencv-python-headless pyyaml

import sys
print(f"Python: {sys.version}")

import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

import nibabel
print(f"NiBabel: {nibabel.__version__}")

import cv2
print(f"OpenCV: {cv2.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Download Public Brain CT Data

This section provides two data source options:

**Option A --- HuggingFace datasets (2D slices):**  
Public brain CT datasets available without authentication. These provide pre-sliced 2D images suitable for quick experimentation.

**Option B --- Synthetic NIfTI volumes (for pipeline testing):**  
Generates synthetic 3D brain CT volumes with realistic ventricle anatomy. These test the full NIfTI-to-YOLO pipeline end-to-end before using real data.

For production training, replace with real NIfTI volumes from:
- [Medical Segmentation Decathlon](http://medicaldecathlon.com/) (Task09_Spleen has CT)
- [TotalSegmentator](https://zenodo.org/records/6802614) datasets
- Your own institutional data with IRB approval

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

os.makedirs('data/raw_nifti', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# ------------------------------------------------------------------
# Option A: Try downloading from HuggingFace (no auth needed)
# ------------------------------------------------------------------
try:
    from datasets import load_dataset
    print("Attempting to download brain CT datasets from HuggingFace...")
    ds = load_dataset("Chanura04/CT-Scan-Brain-Tumor", split="train[:50]")
    print(f"  Loaded {len(ds)} samples from CT-Scan-Brain-Tumor")
    HF_AVAILABLE = True
except Exception as e:
    print(f"  HuggingFace download failed: {e}")
    print("  Falling back to synthetic data generation.")
    HF_AVAILABLE = False

# ------------------------------------------------------------------
# Option B: Create synthetic NIfTI volumes for pipeline testing
# ------------------------------------------------------------------
def create_synthetic_brain_ct(case_id, output_dir='data/raw_nifti'):
    """
    Create a synthetic brain CT NIfTI volume with ventricles.

    The synthetic volume includes:
    - Air background (HU -1000)
    - Skull (HU ~700)
    - Brain parenchyma (HU 25-35)
    - Lateral ventricles (HU ~10), with random size variation
      to simulate normal vs. dilated ventricles (NPH)

    Returns paths to the volume and ventricle mask NIfTI files.
    """
    np.random.seed(case_id)
    shape = (256, 256, 128)

    # Randomize anatomy slightly for each case
    cx = 128 + np.random.randint(-5, 5)
    cy = 128 + np.random.randint(-5, 5)
    vent_scale = 1.0 + np.random.uniform(-0.3, 0.5)  # ventricle size variation

    volume = np.full(shape, -1000, dtype=np.int16)  # air background
    mask = np.zeros(shape, dtype=np.uint8)           # ventricle ground truth

    # Coordinate grids (vectorized for speed)
    x, y, z = np.mgrid[0:shape[0], 0:shape[1], 0:shape[2]]

    # Brain parenchyma (HU 20-40)
    brain_r = ((x - cx) / 90)**2 + ((y - cy) / 100)**2 + ((z - 64) / 44)**2
    brain_mask = brain_r <= 1.0
    volume[brain_mask] = (30 + np.random.randint(-5, 5, size=brain_mask.sum())).astype(np.int16)

    # Skull (HU ~700)
    skull_mask = (brain_r > 0.92) & (brain_r <= 1.15)
    volume[skull_mask] = 700

    # Lateral ventricles (HU ~10) -- bilateral
    lv_l = ((x - (cx - 15)) / (12 * vent_scale))**2 + \
           ((y - cy) / (20 * vent_scale))**2 + \
           ((z - 65) / (18 * vent_scale))**2
    lv_r = ((x - (cx + 15)) / (12 * vent_scale))**2 + \
           ((y - cy) / (20 * vent_scale))**2 + \
           ((z - 65) / (18 * vent_scale))**2
    vent_mask = (lv_l <= 1.0) | (lv_r <= 1.0)
    volume[vent_mask] = 10
    mask[vent_mask] = 1

    # NIfTI affine with typical CT spacing
    spacing = (1.0, 1.0, 1.5)
    affine = np.diag([spacing[0], spacing[1], spacing[2], 1.0])

    vol_path = os.path.join(output_dir, f'brain_ct_{case_id:04d}.nii.gz')
    mask_path = os.path.join(output_dir, f'brain_ct_{case_id:04d}_ventricle_mask.nii.gz')

    nib.save(nib.Nifti1Image(volume, affine), vol_path)
    nib.save(nib.Nifti1Image(mask, affine), mask_path)

    return vol_path, mask_path

# Create 10 synthetic cases
N_SYNTHETIC = 10
print(f"\nCreating {N_SYNTHETIC} synthetic NIfTI volumes for pipeline testing...")
for i in range(N_SYNTHETIC):
    vol_path, mask_path = create_synthetic_brain_ct(i)
    print(f"  Case {i}: {vol_path}")
print(f"\nDone. {N_SYNTHETIC} synthetic cases in data/raw_nifti/")

# ------------------------------------------------------------------
# Quick visualization of one synthetic volume
# ------------------------------------------------------------------
vol_check = nib.load('data/raw_nifti/brain_ct_0000.nii.gz').get_fdata()
mask_check = nib.load('data/raw_nifti/brain_ct_0000_ventricle_mask.nii.gz').get_fdata()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Axial slice at ventricle level
z_mid = vol_check.shape[2] // 2
axes[0].imshow(vol_check[:, :, z_mid], cmap='gray', vmin=-50, vmax=100)
axes[0].set_title(f'Axial Slice z={z_mid} (CT)')
axes[0].axis('off')

axes[1].imshow(mask_check[:, :, z_mid], cmap='Reds', alpha=0.8)
axes[1].set_title(f'Ventricle Mask z={z_mid}')
axes[1].axis('off')

# Overlay
axes[2].imshow(vol_check[:, :, z_mid], cmap='gray', vmin=-50, vmax=100)
axes[2].imshow(mask_check[:, :, z_mid], cmap='Reds', alpha=0.4)
axes[2].set_title(f'Overlay z={z_mid}')
axes[2].axis('off')

plt.suptitle('Synthetic Brain CT Volume -- Case 0000', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nVolume shape: {vol_check.shape}")
print(f"HU range: [{vol_check.min():.0f}, {vol_check.max():.0f}]")
print(f"Ventricle voxels: {mask_check.sum():.0f}")

## 2. Extract Axial Slices as Brain-Windowed PNGs

Brain window settings:
- **Window Level (WL):** 40 HU
- **Window Width (WW):** 80 HU
- **HU range:** [0, 80]

This maps brain parenchyma and CSF to the visible grayscale range while suppressing bone and air.

Each NIfTI volume is reoriented to RAS+ standard before slicing to ensure consistent anatomy across datasets. Slices are resized to 512x512 pixels.

In [ ]:
import cv2
from pathlib import Path

def reorient_to_ras(volume, affine):
    """
    Reorient a 3D volume to RAS+ orientation using its affine matrix.
    Returns the reoriented volume.
    """
    ornt = nib.orientations.io_orientation(affine)
    ras_ornt = nib.orientations.axcodes2ornt(('R', 'A', 'S'))
    if not np.array_equal(ornt, ras_ornt):
        transform = nib.orientations.ornt_transform(ornt, ras_ornt)
        volume = nib.orientations.apply_orientation(volume, transform)
    return volume


def extract_axial_slices(nifti_path, output_dir, window_level=40, window_width=80,
                         target_size=512):
    """
    Extract axial slices from a NIfTI volume as brain-windowed PNGs.

    Parameters
    ----------
    nifti_path : str
        Path to the NIfTI file (.nii or .nii.gz).
    output_dir : str
        Directory to save PNG slices.
    window_level : int
        Center of the HU window (default: 40 for brain).
    window_width : int
        Width of the HU window (default: 80 for brain).
    target_size : int
        Output image size in pixels (default: 512).

    Returns
    -------
    list of (int, str)
        List of (slice_index, png_path) tuples.
    """
    img = nib.load(nifti_path)
    volume = img.get_fdata().astype(np.float32)

    # Reorient to RAS+
    volume = reorient_to_ras(volume, img.affine)

    # Apply brain window
    lower = window_level - window_width / 2
    upper = window_level + window_width / 2
    volume = np.clip(volume, lower, upper)
    volume = ((volume - lower) / (upper - lower) * 255).astype(np.uint8)

    os.makedirs(output_dir, exist_ok=True)
    slice_paths = []

    n_axial = volume.shape[2]
    for z in range(n_axial):
        axial = volume[:, :, z]

        # Resize to target size
        if axial.shape[0] != target_size or axial.shape[1] != target_size:
            axial = cv2.resize(axial, (target_size, target_size),
                               interpolation=cv2.INTER_LINEAR)

        path = os.path.join(output_dir, f'axial_{z:04d}.png')
        cv2.imwrite(path, axial)
        slice_paths.append((z, path))

    return slice_paths


# ------------------------------------------------------------------
# Extract slices from all synthetic NIfTI volumes
# ------------------------------------------------------------------
nifti_files = sorted(Path('data/raw_nifti').glob('brain_ct_*[!k].nii.gz'))
# Filter out mask files
nifti_files = [f for f in nifti_files if 'mask' not in f.name]

print(f"Processing {len(nifti_files)} NIfTI volumes...")
print(f"Brain window: WL={40}, WW={80} -> HU [{0}, {80}]")
print(f"Output size: 512x512 px\n")

all_slices = {}
total_slices = 0

for nifti_path in nifti_files:
    case_id = nifti_path.stem.replace('.nii', '')
    slice_dir = f'data/processed/slices/{case_id}'
    slices = extract_axial_slices(str(nifti_path), slice_dir)
    all_slices[case_id] = slices
    total_slices += len(slices)
    print(f"  {case_id}: {len(slices)} axial slices extracted")

print(f"\nTotal slices extracted: {total_slices}")

# ------------------------------------------------------------------
# Visualize sample extracted slices
# ------------------------------------------------------------------
sample_case = list(all_slices.keys())[0]
sample_slices = all_slices[sample_case]
n_show = min(6, len(sample_slices))
indices = np.linspace(0, len(sample_slices) - 1, n_show, dtype=int)

fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
for i, idx in enumerate(indices):
    z, path = sample_slices[idx]
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'z={z}', fontsize=10)
    axes[i].axis('off')
plt.suptitle(f'Brain-Windowed Axial Slices -- {sample_case}', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Generate YOLO Bounding Box Labels

For each axial slice, we generate YOLO-format bounding box annotations:

**Class 0 -- ventricle:** Derived from the NIfTI ventricle mask. The binary mask is converted to a tight bounding box. Minimum area threshold of 50 pixels prevents noise labels.

**Class 4 -- skull_inner:** Derived from bone thresholding (HU > 300) on the original CT. The bone mask is filled and eroded to approximate the inner skull boundary.

**Classes 1-3 (sylvian_fissure, tight_convexity, pvh):** These require manual annotation and are not auto-generated. Use [Roboflow](https://roboflow.com) or [CVAT](https://cvat.ai) for annotation.

### YOLO Label Format

Each `.txt` label file contains one line per object:
```
class_id center_x center_y width height
```
All values are normalized to [0, 1] relative to image dimensions.

In [ ]:
from scipy.ndimage import binary_fill_holes, binary_erosion


def mask_to_yolo_bbox(mask_2d, class_id, min_area=50):
    """
    Convert a 2D binary mask to YOLO bounding box format.

    Parameters
    ----------
    mask_2d : np.ndarray
        Binary mask (H, W).
    class_id : int
        YOLO class ID.
    min_area : int
        Minimum number of foreground pixels to generate a label.

    Returns
    -------
    str or None
        YOLO format string: "class_id cx cy w h" (normalized), or None.
    """
    if mask_2d.sum() < min_area:
        return None

    rows = np.any(mask_2d, axis=1)
    cols = np.any(mask_2d, axis=0)
    if not rows.any():
        return None

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    h_img, w_img = mask_2d.shape
    cx = ((cmin + cmax) / 2) / w_img
    cy = ((rmin + rmax) / 2) / h_img
    bw = (cmax - cmin) / w_img
    bh = (rmax - rmin) / h_img

    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


def generate_skull_bbox(ct_slice, class_id=4, bone_threshold=300):
    """
    Generate skull_inner bounding box from bone thresholding.

    The bone mask (HU > threshold) is filled and eroded inward
    to approximate the inner skull boundary.
    """
    bone_mask = ct_slice > bone_threshold
    if bone_mask.sum() < 100:
        return None

    filled = binary_fill_holes(bone_mask)
    struct = np.ones((3, 3))
    inner = binary_erosion(filled, structure=struct, iterations=2)

    if not np.any(inner):
        inner = filled

    return mask_to_yolo_bbox(inner.astype(np.uint8), class_id, min_area=100)


def generate_labels_for_case(nifti_vol_path, nifti_mask_path, label_dir,
                             image_size=512):
    """
    Generate YOLO labels for all axial slices of a case.

    Parameters
    ----------
    nifti_vol_path : str
        Path to the CT volume NIfTI.
    nifti_mask_path : str
        Path to the ventricle mask NIfTI.
    label_dir : str
        Output directory for .txt label files.
    image_size : int
        Target image size (must match slice extraction).

    Returns
    -------
    int
        Number of slices that received at least one label.
    """
    vol_img = nib.load(nifti_vol_path)
    vol = vol_img.get_fdata().astype(np.float32)
    mask = nib.load(nifti_mask_path).get_fdata().astype(np.uint8)

    # Reorient both to RAS+ (must match slice extraction orientation)
    vol = reorient_to_ras(vol, vol_img.affine)
    mask = reorient_to_ras(mask, vol_img.affine)

    os.makedirs(label_dir, exist_ok=True)

    n_labeled = 0
    n_axial = vol.shape[2]

    for z in range(n_axial):
        labels = []

        # --- Class 0: ventricle ---
        vent_slice = mask[:, :, z]
        if vent_slice.shape != (image_size, image_size):
            vent_slice = cv2.resize(vent_slice, (image_size, image_size),
                                    interpolation=cv2.INTER_NEAREST)
        vent_bbox = mask_to_yolo_bbox(vent_slice, class_id=0)
        if vent_bbox:
            labels.append(vent_bbox)

        # --- Class 4: skull_inner ---
        ct_slice = vol[:, :, z]
        if ct_slice.shape != (image_size, image_size):
            ct_slice_resized = cv2.resize(ct_slice, (image_size, image_size),
                                          interpolation=cv2.INTER_LINEAR)
        else:
            ct_slice_resized = ct_slice
        skull_bbox = generate_skull_bbox(ct_slice_resized, class_id=4)
        if skull_bbox:
            labels.append(skull_bbox)

        # Write label file (even if empty -- YOLO needs negative examples)
        label_path = os.path.join(label_dir, f'axial_{z:04d}.txt')
        with open(label_path, 'w') as f:
            f.write('\n'.join(labels))

        if labels:
            n_labeled += 1

    return n_labeled


# ------------------------------------------------------------------
# Generate labels for all cases
# ------------------------------------------------------------------
print("Generating YOLO bounding box labels from ventricle masks...\n")

nifti_vols = sorted(Path('data/raw_nifti').glob('brain_ct_*[!k].nii.gz'))
nifti_vols = [f for f in nifti_vols if 'mask' not in f.name]

label_stats = {}
for vol_path in nifti_vols:
    case_id = vol_path.stem.replace('.nii', '')
    mask_path = str(vol_path).replace('.nii.gz', '_ventricle_mask.nii.gz')

    if not os.path.exists(mask_path):
        print(f"  SKIP {case_id} -- no mask file found")
        continue

    label_dir = f'data/processed/labels/{case_id}'
    n = generate_labels_for_case(str(vol_path), mask_path, label_dir)
    label_stats[case_id] = n
    print(f"  {case_id}: {n} slices with bounding boxes")

total_labeled = sum(label_stats.values())
print(f"\nTotal slices with labels: {total_labeled}")

# ------------------------------------------------------------------
# Visualize a sample label overlay
# ------------------------------------------------------------------
sample_case = list(label_stats.keys())[0]
sample_slice_dir = f'data/processed/slices/{sample_case}'
sample_label_dir = f'data/processed/labels/{sample_case}'

# Find a slice with labels
labeled_slice = None
for txt_file in sorted(Path(sample_label_dir).glob('*.txt')):
    content = txt_file.read_text().strip()
    if content:
        labeled_slice = txt_file.stem
        break

if labeled_slice:
    img_path = f'{sample_slice_dir}/{labeled_slice}.png'
    lbl_path = f'{sample_label_dir}/{labeled_slice}.txt'

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    h, w = img.shape

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img, cmap='gray')

    colors = {0: 'cyan', 4: 'yellow'}
    class_names = {0: 'ventricle', 4: 'skull_inner'}

    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:]]

            # Convert normalized to pixel coordinates
            x1 = (cx - bw / 2) * w
            y1 = (cy - bh / 2) * h
            box_w = bw * w
            box_h = bh * h

            color = colors.get(cls_id, 'red')
            rect = plt.Rectangle((x1, y1), box_w, box_h,
                                 linewidth=2, edgecolor=color,
                                 facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, class_names.get(cls_id, f'cls_{cls_id}'),
                    color=color, fontsize=10, fontweight='bold')

    ax.set_title(f'YOLO Labels -- {sample_case} / {labeled_slice}', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"\nSample label file ({lbl_path}):")
    print(Path(lbl_path).read_text())

## 4. Organize into YOLO Dataset Format

The YOLO training format requires a specific directory structure:

```
dataset/
  images/
    train/       # Training images (80% of cases)
    val/         # Validation images (20% of cases)
  labels/
    train/       # Corresponding label .txt files
    val/         # Corresponding label .txt files
  data.yaml      # Dataset configuration
```

**Important:** The train/val split is done **by case** (not by slice) to prevent data leakage. Slices from the same patient volume must all be in the same split.

In [ ]:
import shutil
import random


def organize_yolo_dataset(processed_dir='data/processed', output_dir='dataset',
                          val_split=0.2):
    """
    Organize slices and labels into YOLO training format.

    Split is done BY CASE to prevent data leakage (all slices from
    a single patient volume go into the same split).

    Parameters
    ----------
    processed_dir : str
        Directory containing slices/ and labels/ subdirectories.
    output_dir : str
        Output directory for YOLO dataset.
    val_split : float
        Fraction of cases for validation (default: 0.2).

    Returns
    -------
    str
        Path to the organized dataset directory.
    """
    for split in ['train', 'val']:
        os.makedirs(f'{output_dir}/images/{split}', exist_ok=True)
        os.makedirs(f'{output_dir}/labels/{split}', exist_ok=True)

    # Collect all image-label pairs
    pairs = []
    slice_dirs = sorted(Path(f'{processed_dir}/slices').glob('*'))

    for slice_dir in slice_dirs:
        case_id = slice_dir.name
        label_dir = Path(f'{processed_dir}/labels/{case_id}')

        for img_path in sorted(slice_dir.glob('*.png')):
            label_path = label_dir / img_path.name.replace('.png', '.txt')
            if label_path.exists():
                pairs.append((str(img_path), str(label_path)))

    print(f"Found {len(pairs)} image-label pairs")

    # Split by case (not by slice) to avoid data leakage
    case_ids = sorted(set(Path(p[0]).parent.name for p in pairs))
    random.seed(42)
    random.shuffle(case_ids)
    n_val = max(1, int(len(case_ids) * val_split))
    val_cases = set(case_ids[:n_val])
    train_cases = set(case_ids[n_val:])

    print(f"\nSplit by case ({len(case_ids)} total):")
    print(f"  Train cases ({len(train_cases)}): {sorted(train_cases)}")
    print(f"  Val cases   ({len(val_cases)}):   {sorted(val_cases)}")

    train_count = 0
    val_count = 0

    for img_path, label_path in pairs:
        case_id = Path(img_path).parent.name
        split = 'val' if case_id in val_cases else 'train'

        # Prefix with case_id to avoid name collisions across cases
        fname = f"{case_id}_{Path(img_path).name}"
        lbl_fname = fname.replace('.png', '.txt')

        shutil.copy2(img_path, f'{output_dir}/images/{split}/{fname}')
        shutil.copy2(label_path, f'{output_dir}/labels/{split}/{lbl_fname}')

        if split == 'train':
            train_count += 1
        else:
            val_count += 1

    print(f"\nDataset organized at: {output_dir}/")
    print(f"  Train: {train_count} images")
    print(f"  Val:   {val_count} images")
    print(f"  Total: {train_count + val_count} images")

    return output_dir


dataset_dir = organize_yolo_dataset()

In [ ]:
import yaml

# ------------------------------------------------------------------
# Write dataset.yaml (YOLO configuration file)
# ------------------------------------------------------------------
dataset_config = {
    'path': os.path.abspath('dataset'),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 5,
    'names': {
        0: 'ventricle',
        1: 'sylvian_fissure',
        2: 'tight_convexity',
        3: 'pvh',
        4: 'skull_inner',
    }
}

yaml_path = 'dataset/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"Dataset configuration written to: {yaml_path}")
print()
print("Contents:")
print("-" * 40)
print(Path(yaml_path).read_text())
print("-" * 40)
print()
print("NOTE: Classes 1-3 (sylvian_fissure, tight_convexity, pvh)")
print("require manual annotation using Roboflow or CVAT.")
print("Currently only classes 0 (ventricle) and 4 (skull_inner)")
print("have auto-generated labels.")

## 5. Train YOLOv8 (Optional)

This cell trains a YOLOv8n (nano) model on the prepared dataset. The nano variant is chosen for:
- Fast inference suitable for mobile/edge deployment
- Small model size (~6MB) compatible with on-device processing
- Sufficient accuracy for detection (vs. segmentation) tasks

### CT-Safe Augmentation Settings

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `fliplr` | 0.0 | Left-right flip destroys laterality (clinically meaningful) |
| `flipud` | 0.5 | Up-down flip is safe for axial symmetry |
| `mosaic` | 0.0 | Mosaic destroys spatial anatomy context |
| `mixup` | 0.0 | Mixup destroys spatial anatomy context |
| `hsv_h` | 0.0 | CT is grayscale -- hue is meaningless |
| `hsv_s` | 0.0 | CT is grayscale -- saturation is meaningless |
| `hsv_v` | 0.2 | Small brightness variation is acceptable |
| `degrees` | 5.0 | Slight rotation simulates patient positioning |
| `translate` | 0.1 | Small translation simulates centering variation |
| `scale` | 0.2 | Scale variation simulates different patient sizes |

> **GPU required.** On Google Colab, select Runtime > Change runtime type > T4 GPU.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8n (nano) pretrained on COCO
# For larger datasets, consider yolov8s.pt or yolov8m.pt
model = YOLO('yolov8n.pt')

# Train with CT-safe augmentation constraints
results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=512,
    batch=16,
    device=0,              # GPU (use 'cpu' if no GPU available)
    patience=15,           # Early stopping after 15 epochs without improvement
    project='runs',
    name='nph_yolov8n',
    exist_ok=True,

    # ------ CT-SAFE AUGMENTATIONS ------
    # DO NOT enable fliplr -- laterality is clinically meaningful
    fliplr=0.0,
    # Vertical flip is safe for axial slices
    flipud=0.5,
    # Mosaic and mixup destroy spatial anatomy context
    mosaic=0.0,
    mixup=0.0,
    # CT is grayscale -- hue and saturation are meaningless
    hsv_h=0.0,
    hsv_s=0.0,
    # Small brightness variation is acceptable
    hsv_v=0.2,
    # Geometric augmentations (conservative)
    degrees=5.0,           # Slight rotation (patient positioning)
    translate=0.1,         # Small translation (centering variation)
    scale=0.2,             # Scale variation (patient size)
)

print(f"\nTraining complete.")
print(f"Best model saved to: {results.save_dir}/weights/best.pt")
print(f"Last model saved to: {results.save_dir}/weights/last.pt")

## 6. Validate and Export

Evaluate the trained model on the validation set and export to ONNX format for deployment.

**Metrics to evaluate:**
- **mAP@0.5:** Mean Average Precision at IoU threshold 0.5
- **mAP@0.5:0.95:** Mean Average Precision averaged over IoU thresholds 0.5 to 0.95
- **Precision:** True positives / (True positives + False positives)
- **Recall:** True positives / (True positives + False negatives)

**Export formats:**
- **ONNX:** For browser-based inference (ONNX Runtime Web) and server deployment
- The exported model can be served via FastAPI for the HydroMorph RN app

In [ ]:
# ------------------------------------------------------------------
# Validate the trained model
# ------------------------------------------------------------------
best_model = YOLO('runs/nph_yolov8n/weights/best.pt')
metrics = best_model.val(data='dataset/data.yaml')

print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
print(f"  mAP@0.5:        {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"  Precision:       {metrics.box.mp:.4f}")
print(f"  Recall:          {metrics.box.mr:.4f}")
print("=" * 50)

# Per-class results
class_names = ['ventricle', 'sylvian_fissure', 'tight_convexity', 'pvh', 'skull_inner']
if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:
    print("\nPer-class AP@0.5:")
    for i, ap in enumerate(metrics.box.ap50):
        print(f"  {class_names[i]:20s}: {ap:.4f}")

# ------------------------------------------------------------------
# Export to ONNX for deployment
# ------------------------------------------------------------------
print("\nExporting model to ONNX format...")
onnx_path = best_model.export(format='onnx', imgsz=512)
print(f"ONNX model exported: {onnx_path}")

# Show model file sizes
import glob
print("\nModel files:")
for wt in glob.glob('runs/nph_yolov8n/weights/*'):
    size_mb = os.path.getsize(wt) / (1024 * 1024)
    print(f"  {os.path.basename(wt)}: {size_mb:.1f} MB")

In [ ]:
# ------------------------------------------------------------------
# Quick inference test on a sample image
# ------------------------------------------------------------------
import matplotlib.patches as patches

# Pick a random validation image
val_images = sorted(Path('dataset/images/val').glob('*.png'))
if val_images:
    test_img_path = str(val_images[len(val_images) // 2])

    # Run inference
    results = best_model.predict(test_img_path, conf=0.25, imgsz=512, verbose=False)

    # Visualize
    img = cv2.imread(test_img_path, cv2.IMREAD_GRAYSCALE)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Original image
    axes[0].imshow(img, cmap='gray')
    axes[0].set_title('Input (Brain Window)')
    axes[0].axis('off')

    # Predictions overlay
    axes[1].imshow(img, cmap='gray')

    colors = {0: 'cyan', 1: 'magenta', 2: 'lime', 3: 'orange', 4: 'yellow'}

    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            color = colors.get(cls_id, 'red')
            label = f'{class_names[cls_id]} {conf:.2f}'

            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=2, edgecolor=color,
                                     facecolor='none')
            axes[1].add_patch(rect)
            axes[1].text(x1, y1 - 5, label, color=color,
                         fontsize=9, fontweight='bold',
                         bbox=dict(boxstyle='round,pad=0.2',
                                   facecolor='black', alpha=0.7))

    axes[1].set_title('YOLOv8 Predictions')
    axes[1].axis('off')

    plt.suptitle(f'Inference Test -- {Path(test_img_path).name}', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No validation images found for inference test.")

## 7. Save to Google Drive (Colab)

Optionally save the trained model and dataset to Google Drive for persistence across Colab sessions.

In [ ]:
# ------------------------------------------------------------------
# Save to Google Drive (uncomment to use)
# ------------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
#
# DRIVE_DIR = '/content/drive/MyDrive/hydromorph-nph'
# os.makedirs(DRIVE_DIR, exist_ok=True)
#
# # Copy best model weights
# shutil.copy2('runs/nph_yolov8n/weights/best.pt', f'{DRIVE_DIR}/best.pt')
# shutil.copy2('runs/nph_yolov8n/weights/last.pt', f'{DRIVE_DIR}/last.pt')
#
# # Copy ONNX export
# onnx_files = glob.glob('runs/nph_yolov8n/weights/*.onnx')
# for f in onnx_files:
#     shutil.copy2(f, f'{DRIVE_DIR}/{os.path.basename(f)}')
#
# # Copy dataset config
# shutil.copy2('dataset/data.yaml', f'{DRIVE_DIR}/data.yaml')
#
# print(f"Saved to: {DRIVE_DIR}/")
# print(f"Files: {os.listdir(DRIVE_DIR)}")

print("Uncomment this cell and run to save to Google Drive.")

## 8. Next Steps

### Improve Training Data
- **Add real brain CT volumes:** Replace synthetic data with real NIfTI volumes that have ventricle segmentation ground truth (e.g., from TotalSegmentator or institutional datasets with IRB approval)
- **Manual annotation for classes 1-3:** Use [Roboflow](https://roboflow.com) or [CVAT](https://cvat.ai) to annotate sylvian fissure, tight convexity, and periventricular hyperintensity (PVH) on 50-100 representative slices
- **TotalSegmentator auto-labeling:** Run `TotalSegmentator` on additional CT volumes to auto-generate ventricle masks for class 0

### Model Improvements
- **Scale up:** Try `yolov8s.pt` or `yolov8m.pt` for better accuracy with more training data
- **Segmentation variant:** Consider YOLOv8-seg for instance segmentation instead of bounding boxes
- **Ensemble:** Combine with MedSAM2 predictions for higher confidence

### Deployment
- **HydroMorph RN integration:** The trained `best.pt` can be exported to ONNX and served via FastAPI for the HydroMorph RN app (replacing the current `MockModelProvider.js` for YOLOvx)
- **HuggingFace Spaces:** Deploy alongside the MedSAM2 inference notebook as a Gradio demo
- **On-device:** Export to TFLite or CoreML for on-device inference (requires further optimization)

### References
- [Ultralytics YOLOv8 Documentation](https://docs.ultralytics.com/)
- [TotalSegmentator](https://github.com/wasserth/TotalSegmentator)
- [NiBabel Documentation](https://nipy.org/nibabel/)
- [YOLO Training Tips](https://docs.ultralytics.com/guides/model-training-tips/)